# FIUBench GA-forget5 학습 — Kaggle T4 ×2

논문 Table 7의 GA 설정(`lr=2e-5`, 8 epochs, effective batch 128)을 사용합니다. GA loss는 Forget5 batch만 사용하며 `data_module.py`는 원본 그대로 유지합니다.

Kaggle Settings에서 **GPU T4 ×2**와 Internet을 켜고 순서대로 실행하세요.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao || true
python -m pip install 'transformers==4.37.2' 'tokenizers==0.15.1' 'peft==0.8.2' 'accelerate==1.14.0' 'deepspeed==0.19.7' 'huggingface_hub==0.36.2' 'numpy==2.1.3' hydra-core omegaconf datasets sentencepiece protobuf scipy pillow pyyaml requests tqdm rouge_score scikit-learn


설치 후 커널을 한 번 재시작하고 아래 셀부터 실행하세요.

In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import torch
repo=Path('/kaggle/working/FIUBench_kaggle')
os.chdir(repo)
os.environ['HF_HOME']='/kaggle/working/hf_cache'
expected={'transformers':'4.37.2','tokenizers':'0.15.1','peft':'0.8.2','accelerate':'1.14.0','deepspeed':'0.19.7'}
for package,want in expected.items():
    have=version(package); print(package,have); assert have==want,(package,have)
print('torch:',torch.__version__)
print('GPU count:',torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count()==2


In [ ]:
# 400명 전원의 이미지를 공개 FIUBench dataset revision에서 복원합니다.
import json, shutil, zipfile
from collections import defaultdict
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
repo=Path('/kaggle/working/FIUBench_kaggle')
records=[json.loads(s) for s in (repo/'dataset/full.json').read_text().splitlines() if s.strip()][:400]
needed={r['image_path'] for r in records}
assert len(needed)==400
rev=HfApi().dataset_info('gray311/FIUBench').sha
archive=hf_hub_download('gray311/FIUBench','SFHQ.zip',repo_type='dataset',revision=rev,local_dir='/kaggle/working/fiu_download')
with zipfile.ZipFile(archive) as z:
    entries=defaultdict(list)
    for m in z.infolist():
        p=Path(m.filename)
        if not m.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(m)
    for image_path in needed:
        dest=(repo/image_path).resolve()
        assert dest.is_relative_to((repo/'dataset').resolve())
        hits=entries[dest.name]; assert len(hits)==1,(dest.name,len(hits))
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(hits[0]) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst)
        with Image.open(dest) as im: im.verify()
assert all((repo/p).is_file() for p in needed)
Path('/kaggle/working/fiu_ga_data_revision.txt').write_text(rev)
print('이미지 검증:',len(needed),'dataset revision:',rev)


In [ ]:
# Apply Kaggle-only compatibility patches to forget.py. The GA loss is unchanged.
import ast
from pathlib import Path
path=Path('/kaggle/working/FIUBench_kaggle/forget.py')
text=path.read_text()
original=text
old='AutoTokenizer.from_pretrained(cfg.model_path)'
new='AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)'
assert old in text or new in text
text=text.replace(old,new)
marker='    if cfg.LoRA.r != 0:\n        config = LoraConfig('
block='''    if str(model_cfg.get("gradient_checkpointing", "false")).lower() == "true":
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        logger.info("Gradient checkpointing enabled", main_process_only=True)

'''
if 'model.gradient_checkpointing_enable()' not in text:
    assert text.count(marker)==1
    text=text.replace(marker,block+marker,1)
end=text.rfind('\nif __name__ == "__main__":')
assert end>0
if 'if cfg.save_model and accelerator.is_main_process:' not in text:
    start=text.rfind('\n    accelerator.end_training()',0,end)
    assert start>0
    replacement='''
    output_dir = cfg.save_dir
    accelerator.wait_for_everyone()
    if cfg.save_model and accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
        unwrapped_model = accelerator.unwrap_model(model)
        if cfg.LoRA.r != 0:
            save_lora_weights(unwrapped_model, output_dir)
        else:
            unwrapped_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
    accelerator.wait_for_everyone()
    accelerator.end_training()
'''
    text=text[:start]+replacement+text[end:]
ast.parse(text)
backup=path.with_suffix('.py.before_kaggle_patch')
if not backup.exists(): backup.write_text(original)
path.write_text(text)
assert 'use_fast=False' in text and 'model.enable_input_require_grads()' in text
print('Kaggle compatibility patch complete')


In [ ]:
# GA Forget5 config와 T4×2 Accelerate config를 명시적으로 작성합니다.
from pathlib import Path
repo=Path('/kaggle/working/FIUBench_kaggle')
(repo/'config/forget_ga_forget5.yaml').write_text("""defaults:
  - forget_lora
  - _self_
model_family: llava-phi
model_path: gray311/vlm_unlearning_ft_llava_phi_3_mini
attn_implementation: sdpa
data_path: ./dataset/full.json
split: forget5
forget_loss: ga
lr: 2e-5
batch_size: 4
gradient_accumulation_steps: 16
num_epochs: 8
LoRA: {r: 128, alpha: 256, dropout: 0.05}
workers: 0
report_to: null
save_dir: /kaggle/working/models/llava_phi_GA_forget5
save_model: true
save_steps: 4
""")
(repo/'config/accelerate_kaggle_t4x2.yaml').write_text("""compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 16
  gradient_clipping: 1.0
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: false
  zero_stage: 2
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
""")
print((repo/'config/forget_ga_forget5.yaml').read_text())


In [ ]:
# GA가 사용하는 Forget5 크기와 optimizer step 수를 검증합니다.
import json
from pathlib import Path
split=json.loads(Path('dataset/split.json').read_text())
rows=[json.loads(x) for x in Path('dataset/full.json').read_text().splitlines() if x.strip()][:400]
forget=set(split['forget5'])
forget_qa=sum(len(r['qa_list']) for r in rows if r['unique_id'] in forget)
print('forget identities:',len(forget),'forget QA:',forget_qa)
assert len(forget)==20 and forget_qa==400
print('effective forget batch:',4*2*16,'총 optimizer steps:',32)
print('GA loss는 retain_inputs를 사용하지 않습니다.')


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/FIUBench_kaggle
export HF_HOME=/kaggle/working/hf_cache
export TOKENIZERS_PARALLELISM=false
export HYDRA_FULL_ERROR=1
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA_VISIBLE_DEVICES=0,1 DS_SKIP_CUDA_CHECK=1 accelerate launch \
  --config_file config/accelerate_kaggle_t4x2.yaml \
  --main_process_port 2233 \
  forget.py --config-name forget_ga_forget5 \
  2>&1 | tee /kaggle/working/ga_forget5_console.log
echo TRAIN_PROCESS_EXIT_0


In [ ]:
# 산출물을 검증합니다.
from pathlib import Path
import torch
out=Path('/kaggle/working/models/llava_phi_GA_forget5')
print('중간 checkpoints:',sorted(str(p.parent) for p in out.glob('step_*/checkpoint.pt')))
ckpt=out/'checkpoint.pt'
assert ckpt.is_file(),ckpt
state=torch.load(ckpt,map_location='cpu',weights_only=True)
print('최종 checkpoint:',ckpt,'MiB:',round(ckpt.stat().st_size/1024**2,1),'tensor 수:',len(state))
print('다음 평가에서는 step_4부터 step_32까지 같은 설정으로 비교하세요.')


학습이 끝나면 `/kaggle/working/models/llava_phi_GA_forget5`, `ga_forget5_console.log`, `fiu_ga_data_revision.txt`를 Save Version으로 보존하세요. 논문의 early-stopping 규칙이 공개되지 않았으므로 최종 step만 보지 말고 step 4 간격 체크포인트를 동일 평가로 비교합니다.